In [18]:
pip install -U langchain-text-splitters

In [16]:
!pip install langchain langchain-community langchain-core langchain.text_splitter
!pip install chromadb sentence-transformers
!pip install pypdf
!pip install langgraph

ERROR: Could not find a version that satisfies the requirement langchain.text_splitter (from versions: none)
ERROR: No matching distribution found for langchain.text_splitter


In [25]:
from google.colab import files
uploaded = files.upload()

Saving Technical Documentation.pdf to Technical Documentation.pdf


In [26]:
!ls /content


 LLD.pdf      "Soniya's_research_proposal.pdf"
 sample_data  'Technical Documentation.pdf'


In [27]:
file_path = "/content/Technical Documentation.pdf"

In [41]:
from transformers import pipeline

# Use a small GPT model (compatible)
generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=150
)

# Query
query = "What is a workflow ?"

# Retrieve relevant chunks
docs = retriever.invoke(query)

# Combine context
context = "\n".join([doc.page_content for doc in docs[:3]])

# Prompt
prompt = f"""
Context:
{context}

Question: {query}
Answer:
"""

# Generate response
response = generator(prompt)

print(response[0]['generated_text'])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Context:
LOW LEVEL DESIGN (LLD) 
 
1. Modules 
• Document Processing Module  
• Chunking Module  
• Embedding Module  
• Vector Storage Module  
• Retrieval Module  
• Query Processing Module  
• Graph Execution Module  
• HITL Module 
2. Data Structures 
• Document → Raw text  
• Chunk → Text segment  
• Embedding → Vector  
• Query → Input string  
• State → Workflow data 
3. Workflow Design
Technical  Documentation  
Introduction  
RAG  combines  retrieval  and  generation  to  provide  accurate  responses.  It  improves  
reliability
 
over
 
standalone
 
LLMs.
 
System  Architecture  
The  system  integrates  ChromaDB,  HuggingFace  embeddings,  and  LangGraph  workflow  
for
 
decision-making.
 
Design  Decisions  
●  Chunk  size:  500  (balance  context  +  speed)  ●  Embedding:  all-MiniLM-L6-v2  ●  Retrieval:  similarity  search  
Workflow  
LangGraph  orchestrates  nodes:  ●  Processing  Node  ●  Decision  Node  ●  Output  Node  
Conditional  Logic  
Routing  is  based  on  

In [42]:
from langgraph.graph import StateGraph

class State(dict):
    pass

# Node 1: Processing
def process_node(state):
    # ✅ Force safe extraction
    query = state.get("query", "")

    if not query:
        return {"answer": "No query provided."}

    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs[:3]])

    prompt = f"""
    Context:
    {context}

    Question: {query}
    Answer:
    """

    response = generator(prompt)
    answer = response[0]['generated_text']

    return {"answer": answer}


# Node 2: Output
def output_node(state):
    print("Final Answer:\n", state.get("answer", "No answer"))
    return state


# Build graph
graph = StateGraph(State)

graph.add_node("process", process_node)
graph.add_node("output", output_node)

graph.set_entry_point("process")
graph.add_edge("process", "output")

app = graph.compile()

# ✅ Important: wrap input properly
app.invoke(State({"query": "What is a workflow?"}))

Final Answer:
 No answer


In [43]:
from langgraph.graph import StateGraph

class State(dict):
    pass

# Node 1: Processing
def process_node(state):
    # Always extract the query
    query = state.get("query", "")
    if not query:
        return {"answer": "No query provided."}

    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs[:3]])
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"

    response = generator(prompt)
    answer = response[0]['generated_text']

    # CRITICAL: Return both the query and the answer to maintain state
    return {"query": query, "answer": answer}

# Node 2: Output
def output_node(state):
    # Now 'answer' will be present in the state
    answer = state.get("answer", "No answer found")
    print(f"Final Answer:\n{answer}")
    return {"answer": answer}

# Build graph
graph = StateGraph(State)

graph.add_node("process", process_node)
graph.add_node("output", output_node)

graph.set_entry_point("process")
graph.add_edge("process", "output")

app = graph.compile()

# ✅ Important: wrap input properly
app.invoke(State({"query": "What is a workflow?"}))

Final Answer:
No answer found
